In [ ]:
import pandas as pd
import geopandas as gpd

In [26]:
df_geo = pd.read_csv("../data/raw/countries.csv")
print(df_geo.info())

<class 'pandas.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country         176 non-null    str    
 1   Land Area(Km2)  175 non-null    float64
 2   Latitude        175 non-null    float64
 3   Longitude       175 non-null    float64
dtypes: float64(3), str(1)
memory usage: 5.6 KB
None


In [27]:
print(df_geo.isna().sum())

Country           0
Land Area(Km2)    1
Latitude          1
Longitude         1
dtype: int64


In [28]:
print(df_geo[df_geo.isna().any(axis=1)])

          Country  Land Area(Km2)  Latitude  Longitude
58  French Guiana             NaN       NaN        NaN


In [29]:
df_geo.loc[df_geo['Country'] == 'French Guiana', 'Land Area(Km2)'] = 83534
df_geo.loc[df_geo['Country'] == 'French Guiana', 'Latitude'] = 3.9339
df_geo.loc[df_geo['Country'] == 'French Guiana', 'Longitude'] = -53.1258

In [30]:
print(df_geo.isna().sum())

Country           0
Land Area(Km2)    0
Latitude          0
Longitude         0
dtype: int64


In [31]:
print(df_geo.describe())

       Land Area(Km2)    Latitude   Longitude
count    1.760000e+02  176.000000  176.000000
mean     6.279038e+05   18.220214   14.514935
std      1.580423e+06   24.150639   66.301570
min      2.100000e+01  -40.900557 -175.198242
25%      2.618175e+04    3.740173  -12.448007
50%      1.151110e+05   17.125346   19.259763
75%      5.168320e+05   38.995841   45.359275
max      9.984670e+06   64.963051  178.065032


In [32]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
gdf = gpd.read_file(url)
print(gdf.head())

        featurecla  scalerank  LABELRANK                   SOVEREIGNT SOV_A3  \
0  Admin-0 country          1          6                         Fiji    FJI   
1  Admin-0 country          1          3  United Republic of Tanzania    TZA   
2  Admin-0 country          1          7               Western Sahara    SAH   
3  Admin-0 country          1          2                       Canada    CAN   
4  Admin-0 country          1          2     United States of America    US1   

   ADM0_DIF  LEVEL               TYPE TLC                        ADMIN  ...  \
0         0      2  Sovereign country   1                         Fiji  ...   
1         0      2  Sovereign country   1  United Republic of Tanzania  ...   
2         0      2      Indeterminate   1               Western Sahara  ...   
3         0      2  Sovereign country   1                       Canada  ...   
4         1      2            Country   1     United States of America  ...   

      FCLASS_TR     FCLASS_ID     FCLASS_PL 

In [33]:
world_gdf = gdf[['ADMIN', 'CONTINENT', 'geometry']].copy()
world_gdf.rename(columns={'ADMIN': 'Country', 'CONTINENT': 'Continent'}, inplace=True)

# Ensure geometry column is valid
world_gdf = gpd.GeoDataFrame(world_gdf, geometry='geometry', crs="EPSG:4326")

# Check geometry types
print(world_gdf['geometry'].geom_type.value_counts())

Polygon         148
MultiPolygon     29
Name: count, dtype: int64


In [34]:
country_mapping = {
    "Antigua and Barbuda": "Antigua and Barb.",
    "Aruba": "Aruba",
    "Bahamas": "The Bahamas",
    "Bahrain": "Bahrain",
    "Barbados": "Barbados",
    "Bermuda": "Bermuda",
    "Cayman Islands": "Cayman Is.",
    "Comoros": "Comoros",
    "Congo": "Republic of the Congo",
    "Dominica": "Dominican Republic",
    "Eswatini": "eSwatini",
    "French Guiana": "French Guiana",
    "Grenada": "Grenada",
    "Kiribati": "Kiribati",
    "Maldives": "Maldives",
    "Malta": "Malta",
    "Mauritius": "Mauritius",
    "Nauru": "Nauru",
    "Saint Kitts and Nevis": "Saint Kitts and Nevis",
    "Saint Lucia": "Saint Lucia",
    "Saint Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "Samoa": "Samoa",
    "Sao Tome and Principe": "Sao Tome and Principe",
    "Serbia": "Republic of Serbia",
    "Seychelles": "Seychelles",
    "Singapore": "Singapore",
    "Tonga": "Tonga",
    "Tuvalu": "Tuvalu",
    "United States": "United States of America",
}
df_geo["Country_corrected"] = df_geo["Country"].replace(country_mapping)

In [35]:
# Check remaining missing
missing_after = [c for c in df_geo['Country_corrected'].unique() if c not in world_gdf['Country'].values]
print("Still missing:", missing_after)

Still missing: ['Antigua and Barb.', 'Aruba', 'Bahrain', 'Barbados', 'Bermuda', 'Cayman Is.', 'Comoros', 'French Guiana', 'Grenada', 'Kiribati', 'Maldives', 'Malta', 'Mauritius', 'Nauru', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Samoa', 'Sao Tome and Principe', 'Seychelles', 'Singapore', 'Tonga', 'Tuvalu']


In [38]:
# Assume your df has a column 'Country_corrected' that matches world_gdf['Country']
df_merged = df_geo.merge(
    world_gdf[['Country', 'Continent', 'geometry']],
    left_on='Country_corrected',
    right_on='Country',
    how='left'  # keeps all countries even if not found in world_gdf
)

# Convert to GeoDataFrame (geometry may have NaNs for missing countries)
df_merged = gpd.GeoDataFrame(df_merged, geometry='geometry', crs="EPSG:4326")

# Optional: preview
print(df_merged.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Country_x          176 non-null    str     
 1   Land Area(Km2)     176 non-null    float64 
 2   Latitude           176 non-null    float64 
 3   Longitude          176 non-null    float64 
 4   Country_corrected  176 non-null    str     
 5   Country_y          153 non-null    str     
 6   Continent          153 non-null    str     
 7   geometry           153 non-null    geometry
dtypes: float64(3), geometry(1), str(4)
memory usage: 11.1 KB
None


In [39]:
# Drop the unnecessary columns
df_merged_clean = df_merged.drop(columns=['Country_corrected', 'Country_y'])

# Rename Country_x to Country
df_merged_clean.rename(columns={'Country_x': 'Country'}, inplace=True)

# Preview
print(df_merged_clean.head())

               Country  Land Area(Km2)   Latitude  Longitude Continent  \
0          Afghanistan        652230.0  33.939110  67.709953      Asia   
1              Albania         28748.0  41.153332  20.168331    Europe   
2              Algeria       2381741.0  28.033886   1.659626    Africa   
3               Angola       1246700.0 -11.202692  17.873887    Africa   
4  Antigua and Barbuda           443.0  17.060816 -61.796428       NaN   

                                            geometry  
0  POLYGON ((66.51861 37.36278, 67.07578 37.35614...  
1  POLYGON ((21.02004 40.84273, 20.99999 40.58, 2...  
2  POLYGON ((-8.6844 27.39574, -8.66512 27.58948,...  
3  MULTIPOLYGON (((12.99552 -4.7811, 12.63161 -4....  
4                                               None  


In [40]:
# Countries with missing Continent
missing_continent = df_merged_clean[df_merged_clean['Continent'].isna()]['Country']

print("Countries with missing Continent:")
print(missing_continent.tolist())

Countries with missing Continent:
['Antigua and Barbuda', 'Aruba', 'Bahrain', 'Barbados', 'Bermuda', 'Cayman Islands', 'Comoros', 'French Guiana', 'Grenada', 'Kiribati', 'Maldives', 'Malta', 'Mauritius', 'Nauru', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Samoa', 'Sao Tome and Principe', 'Seychelles', 'Singapore', 'Tonga', 'Tuvalu']


In [42]:
# Manual continent assignment for missing countries
manual_continent_map = {
    "Antigua and Barbuda": "North America",
    "Aruba": "North America",
    "Bahrain": "Asia",
    "Barbados": "North America",
    "Bermuda": "North America",
    "Cayman Islands": "North America",
    "Comoros": "Africa",
    "French Guiana": "South America",
    "Grenada": "North America",
    "Kiribati": "Oceania",
    "Maldives": "Asia",
    "Malta": "Europe",
    "Mauritius": "Africa",
    "Nauru": "Oceania",
    "Saint Kitts and Nevis": "North America",
    "Saint Lucia": "North America",
    "Saint Vincent and the Grenadines": "North America",
    "Samoa": "Oceania",
    "Sao Tome and Principe": "Africa",
    "Seychelles": "Africa",
    "Singapore": "Asia",
    "Tonga": "Oceania",
    "Tuvalu": "Oceania",
}

# Fill the missing continents
df_merged_clean["Continent"] = df_merged_clean.apply(
    lambda row: (
        manual_continent_map[row["Country"]]
        if pd.isna(row["Continent"]) and row["Country"] in manual_continent_map
        else row["Continent"]
    ),
    axis=1,
)

# Check if all continents are now filled
print(df_merged_clean[df_merged_clean["Continent"].isna()])

Empty GeoDataFrame
Columns: [Country, Land Area(Km2), Latitude, Longitude, Continent, geometry]
Index: []


In [ ]:
from shapely.geometry import Point

# Fill missing geometry with Point(lon, lat)
df_merged_clean["geometry"] = df_merged_clean.apply(
    lambda row: (
        Point(row["Longitude"], row["Latitude"])
        if row["geometry"] is None
        else row["geometry"]
    ),
    axis=1,
)

# Now convert geometry to WKT safely
df_merged_clean["geometry_wkt"] = df_merged_clean["geometry"].apply(
    lambda g: g.wkt if g is not None else None
)

# Save to CSV
df_merged_clean.to_csv("../data/processed/country_geo.csv", index=False)